## LLM Setup

In [1]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    temperature=0,
    groq_api_key=api_key,
    model_name="llama-3.3-70b-versatile"
)

# Adding the JSON Parser
parser = JsonOutputParser()

C:\Users\Administrator\Desktop\Career-Assistant-AI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Scraping a job posting page

In [2]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://kenya.ai/opportunities/lead-data-scientist-at-kyosk-digital-services/?utm_campaign=google_jobs_apply&utm_source=google_jobs_apply&utm_medium=organic")
page_data = loader.load().pop().page_content
# print(page_data)

USER_AGENT environment variable not set, consider setting it to identify your requests.


## Extracting job data

In [3]:
prompt_extract = PromptTemplate.from_template(
"""
You are an expert information extraction system.

### CONTEXT
The following text was scraped from a company's career page and contains
information about a job posting. The text may include formatting noise,
navigation elements, or repeated content.

### TASK
Extract the job information from the text.

Return a JSON object with the following fields:

- role: The job title
- experience: Required experience level (years or level such as junior, mid, senior)
- skills: List of key technical or domain skills required for the role
- description: A concise 2–5 sentence summary of the job responsibilities

### RULES
- Extract information only from the provided text.
- Do NOT invent information.
- If a field is missing, return null.
- Skills must be returned as a list of strings.
- Return ONLY a valid JSON object.
- Do not include explanations, markdown, backticks, or extra text

### JSON FORMAT
{{
  "role": "Job title",
  "experience": "Experience requirement",
  "skills": ["skill1", "skill2", "skill3"],
  "description": "Short description of the role"
}}

### SCRAPED TEXT
{page_data}
"""
)

# Building the extract chain
chain_extract = prompt_extract | llm | parser

In [4]:
# Running the chain extraction on the scraped jd
job_data = chain_extract.invoke({"page_data": page_data})
print(job_data)

{'role': 'Lead Data Scientist', 'experience': 'Minimum of 3 years', 'skills': ['R', 'Python', 'SQL', 'Java', 'Scala', 'C++', 'JavaScript', 'Machine learning', 'Data mining', 'Data visualization', 'Statistical analysis', 'Looker Studio'], 'description': 'The Lead Data Scientist will be responsible for discovering insights from complex data sets and building models to support business objectives and decision-making. The role holder will provide new insights into the business and utilize advanced statistical analysis, data mining, and data visualization techniques to create solutions that enable enhanced business performance. The Lead Data Scientist will work with stakeholders to identify opportunities for leveraging company data to drive business solutions and lead project execution to ensure that data projects are completed on time and according to service delivery specifications.'}


## Extracting resume data

In [6]:
import pdfplumber

def read_resume(path="resume.pdf"):
    text = ""
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text

resume_text = read_resume()
print(resume_text[:1000])

Sharlyn Muturi
Portfolio:sharlynmuturi.github.io
(+254)748-736720 github.com/sharlynmuturi
DataScientist/AIEngineer/Underwriter
sharlynnmuturi@gmail.com
Analyticalactuarialsciencegraduatewithexperienceininsuranceoperations,claimsanalytics,policyevaluation,andrisk
assessment.Strongbackgroundinstatisticalmodelingandmachinelearning.ProficientinPython,R,SQL,andDjangoORMfor
dataanalysisanddata-drivenapplicationdesign,withexperiencebuildingLLM-poweredapplications.Interestedinapplyingdata
andAItodevelopscalablesolutionsthatsupportinformeddecision-makinginanalyticsandrisk-focusedenvironments.
SKILLS
ProgrammingandWeb Python,R,SQL,JavaScript,HTML/CSS,Django,Flask,Streamlit.
Data&Analytics Databricks,Tableau,PowerBI,SQLServer,MySQL,PostgreSQL,Spark,Excel,Access,A/Btesting.
AIEngineering PromptEngineering,LangChain,RAG,OpenAI,Groq,HuggingFace,GenAI,AgenticAI,DocumentAI.
MachineLearning RegressionandClassificationModels,TimeSeriesAnalysis,PredictiveModeling,NLP,NeuralNetworks.
InsuranceOperations 

In [7]:
prompt_resume = PromptTemplate.from_template(
"""
You are an expert resume parser.

### TASK
Extract structured information from the resume text.

Return a JSON object with the following fields:

- name: candidate name
- education: list of education entries
- skills: list of technical skills
- experience: list of professional experiences
- projects: list of projects mentioned

### RULES
- Extract information only from the text.
- Do NOT invent information.
- If something is missing return null.
- Skills must be a list of strings.
- Return ONLY valid JSON.

### JSON FORMAT
{{
  "name": "Candidate name",
  "education": ["degree1", "degree2"],
  "skills": ["skill1", "skill2"],
  "experience": ["experience1", "experience2"],
  "projects": ["project1", "project2"]
}}

### RESUME TEXT
{resume_text}
"""
)

parser = JsonOutputParser()

chain_resume = prompt_resume | llm | parser

# Running the resume extraction
resume_data = chain_resume.invoke({"resume_text": resume_text})

print(resume_data)

{'name': 'Sharlyn Muturi', 'education': ['Bachelor of Science in Actuarial Science, GPA: 3.5, The University of Nairobi 2020—2024', 'Kenya Certificate of Secondary Education (K.C.S.E), Grade: A-, Chogoria Girls’ High School 2016—2019'], 'skills': ['Python', 'R', 'SQL', 'JavaScript', 'HTML/CSS', 'Django', 'Flask', 'Streamlit', 'Databricks', 'Tableau', 'PowerBI', 'SQL Server', 'MySQL', 'PostgreSQL', 'Spark', 'Excel', 'Access', 'Prompt Engineering', 'LangChain', 'RAG', 'OpenAI', 'Groq', 'HuggingFace', 'GenAI', 'AgenticAI', 'DocumentAI'], 'experience': ['UNDERWRITING ASSISTANT Dec 2024—June 2025, Incourage Insurance Agency Nairobi, Kenya', 'INSURANCE OFFICER ATTACHEE May 2023—Jul 2023, Kenya Electricity Generating Company PLC (KenGen) Nairobi, Kenya'], 'projects': ['JiBudget - Personal finance and budgeting web application', 'Motor Insurance System - System supporting policy issuance, underwriting workflows and data storage', 'M-PESA Transactions Analytics - Platform that analyzes M-PESA s

In [8]:
print("\nCandidate:", resume_data["name"])
print("\nSkills:", ", ".join(resume_data["skills"]))
print("\nEducation:", resume_data["education"])
print("\nProjects:", resume_data["projects"])


Candidate: Sharlyn Muturi

Skills: Python, R, SQL, JavaScript, HTML/CSS, Django, Flask, Streamlit, Databricks, Tableau, PowerBI, SQL Server, MySQL, PostgreSQL, Spark, Excel, Access, Prompt Engineering, LangChain, RAG, OpenAI, Groq, HuggingFace, GenAI, AgenticAI, DocumentAI

Education: ['Bachelor of Science in Actuarial Science, GPA: 3.5, The University of Nairobi 2020—2024', 'Kenya Certificate of Secondary Education (K.C.S.E), Grade: A-, Chogoria Girls’ High School 2016—2019']

Projects: ['JiBudget - Personal finance and budgeting web application', 'Motor Insurance System - System supporting policy issuance, underwriting workflows and data storage', 'M-PESA Transactions Analytics - Platform that analyzes M-PESA statements using regex and LLMs', 'AI Resume & Cover Letter Tailoring - AI assistant using RAG, LangChain, ChromaDB and Groq', 'Portfolio Optimization - Combining Prophet forecasts with Modern Portfolio Theory and AgenticAI', 'SAAS Customer Churn - Predicting churn probability 

In [9]:
job_skills = set(job_data["skills"])
resume_skills = set(resume_data["skills"])

matching = job_skills.intersection(resume_skills)

print("Matching skills:", matching)

Matching skills: {'SQL', 'Python', 'R', 'JavaScript'}


## Embedding Portfolio Projects

In [12]:
import pandas as pd
import uuid
import chromadb

In [13]:
portfolio_df = pd.read_csv("portfolio.csv")
portfolio_df.head(2)

,project_name,description,tech_stack,link,source_page,all_text
0,JiBudget,Personal finance and budgeting web application...,"bootstrap, css, django, html, html/css, javasc...",https://github.com/sharlynmuturi/JiBudget,https://sharlynmuturi.github.io/,JiBudget Personal finance and budgeting web ap...
1,Motor Insurance Database,Database-driven insurance management system su...,"bootstrap, css, django, html, html/css, mysql,...",https://github.com/sharlynmuturi/motor-insuran...,https://sharlynmuturi.github.io/,Motor Insurance Database Database-driven insur...


In [14]:
# ChromaDB for portfolio
client = chromadb.PersistentClient('vectorstore')
collection = client.get_or_create_collection(name="portfolio")

if not collection.count():
    for _, row in portfolio_df.iterrows():
        collection.add(
            documents=[row["all_text"]],
            metadatas={
                "link": row.get("link",""),
                "project_name": row.get("project_name",""),
                "tech_stack": row.get("tech_stack","")
            },
            ids=[str(uuid.uuid4())]
        )

print(f"Total projects in ChromaDB: {collection.count()}")

Total projects in ChromaDB: 20


In [15]:
# Retrieving Portfolio Projects
job_description_text = job_data["description"]

# Querying chromaDB for relevant projects
results = collection.query(
    query_texts=[job_description_text],
    n_results=5  # top 5 most relevant projects
)

# Format top projects for the LLM
top_projects_text = ""
for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
    project_name = meta.get("project_name", "Unknown Project")
    tech_stack = meta.get("tech_stack", "")  # empty string if missing
    link = meta.get("link", "")

    top_projects_text += f"""
Project: {project_name}
Description: {doc}
Technologies: {tech_stack}
Link: {link}
"""

print(top_projects_text[:800])


Project: HR Data Analysis & Visualization
Description: HR Data Analysis & Visualization Exploratory data analysis and building interactive dashboards to analyze workforce and organizational metrics. analysis
Technologies: analysis
Link: https://github.com/sharlynmuturi/data-engineering-and-analytics-projects/tree/main/customer-shopping-behavior-analysis

Project: Product & Customer Sales Performance ETL
Description: Product & Customer Sales Performance ETL End-to-end customer and product sales analytics pipeline, including data ingestion, aggregation, feature engineering, performance analysis and visualization. analysis, analytics, visualization
Technologies: analysis, analytics, visualization
Link: https://github.com/sharlynmuturi/data-engineering-and-analytics-projects/tree/main/custome


## Resume and Cover Letter Tailoring

In [16]:
prompt_tailor_resume = PromptTemplate.from_template(
"""
You are an expert career assistant and resume writer.

Your task is to tailor a candidate's resume for a specific job role.

### JOB DESCRIPTION
{job_data}

### CANDIDATE RESUME
{resume_data}

### CANDIDATE PROJECTS
{portfolio_projects}

### INSTRUCTIONS
Rewrite the candidate's resume so that it better aligns with the job.

Focus on:
- highlighting relevant skills
- emphasizing relevant projects
- matching the language used in the job description
- keeping all information truthful

### OUTPUT FORMAT

Return a professional resume with the following sections:

NAME

PROFESSIONAL SUMMARY

SKILLS

EXPERIENCE

PROJECTS

EDUCATION

The resume should be concise, professional, and optimized for ATS systems.

Return only the resume text.
"""
)

chain_tailor_resume = prompt_tailor_resume | llm

In [17]:
tailored_resume = chain_tailor_resume.invoke({
    "job_data": job_data,
    "resume_data": resume_text,
    "portfolio_projects": top_projects_text
})

print("TAILORED RESUME\n")
print(tailored_resume.content)

TAILORED RESUME

Sharlyn Muturi

PROFESSIONAL SUMMARY
Results-driven Lead Data Scientist with 3+ years of experience in discovering insights from complex data sets and building models to support business objectives and decision-making. Proficient in machine learning, data mining, and data visualization, with a strong background in statistical analysis and programming languages such as Python, R, SQL, and Java.

SKILLS
- Programming languages: Python, R, SQL, Java, JavaScript
- Data analysis and visualization: Tableau, PowerBI, Databricks, SQL Server, MySQL, PostgreSQL, Spark
- Machine learning: Regression and Classification Models, Time Series Analysis, Predictive Modeling, NLP, Neural Networks
- Data mining and statistical analysis: Data visualization, Statistical analysis, Data mining
- Operating Systems: Windows, Linux

EXPERIENCE
UNDERWRITING ASSISTANT
Incourage Insurance Agency, Nairobi, Kenya
Dec 2024 - June 2025
- Analyzed risk data, including transactional and premium reconcili

In [18]:
prompt_cover_letter = PromptTemplate.from_template("""
You are an expert career assistant.

### CONTEXT
You have the following information:

- Job posting details: {job_data}
- Candidate resume content: {resume_data}
- Relevant portfolio projects: {portfolio_projects}

### TASK
Write a professional and persuasive **cover letter** tailored to this specific job. 
The cover letter should:

1. Address the hiring manager (use "Dear Hiring Manager" if name is unknown)
2. Highlight the candidate's most relevant skills and experience from the resume
3. Reference the most relevant portfolio projects
4. Match the tone of the job posting (formal, professional)
5. Be 3–5 paragraphs long
6. End with a polite call-to-action

### RULES
- Only use information provided in the context; do not invent details
- Keep it concise and impactful
- Output plain text (no JSON)
""")

chain_cover_letter = prompt_cover_letter | llm

res_cover_letter = chain_cover_letter.invoke({
    "job_data": job_data,
    "resume_data": resume_text,
    "portfolio_projects": top_projects_text
})

# output
cover_letter_text = res_cover_letter["content"] if isinstance(res_cover_letter, dict) else res_cover_letter.content

print("AI GENERATED COVER LETTER \n")
print(cover_letter_text)

AI GENERATED COVER LETTER 

Dear Hiring Manager,

I am excited to apply for the Lead Data Scientist position at your esteemed organization. With a strong background in statistical modeling and machine learning, I am confident in my ability to discover insights from complex data sets and build models to support business objectives and decision-making. My experience in insurance operations, claims analytics, and policy evaluation has equipped me with a unique understanding of the importance of data-driven decision-making in driving business solutions.

As a skilled data scientist with proficiency in Python, R, SQL, and Django ORM, I have a proven track record of building data-driven applications and analyzing complex data sets. My portfolio projects demonstrate my expertise in data analysis, machine learning, and data visualization. For instance, my HR Data Analysis & Visualization project showcases my ability to perform exploratory data analysis and build interactive dashboards to analy

## Semantic Matching

In [19]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Embedding job
description = job_data.get("description") or ""
skills_list = job_data.get("skills") or []
job_text = description + " " + " ".join(skills_list)

job_embedding = embedding_model.embed_documents([job_text])[0]

Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 2135.61it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [31]:
# Embedding portfolio projects
project_texts = portfolio_df['all_text'].tolist()
project_embeddings = embedding_model.embed_documents(project_texts)

In [32]:
# Computing cosine similarity
from numpy import dot
from numpy.linalg import norm

similarities = [dot(job_embedding, p_emb) / (norm(job_embedding) * norm(p_emb)) for p_emb in project_embeddings]

In [33]:
# Get top 5 projects
top_indices = sorted(range(len(similarities)), key=lambda i: similarities[i], reverse=True)[:5]
top_semantic_projects = portfolio_df.iloc[top_indices]
top_semantic_projects

,project_name,description,tech_stack,link,source_page,all_text
16,HR Data Analysis & Visualization,Exploratory data analysis and building interac...,analysis,https://github.com/sharlynmuturi/data-engineer...,https://sharlynmuturi.github.io/elements.html,HR Data Analysis & Visualization Exploratory d...
13,Product & Customer Sales Performance ETL,End-to-end customer and product sales analytic...,"analysis, analytics, visualization",https://github.com/sharlynmuturi/data-engineer...,https://sharlynmuturi.github.io/elements.html,Product & Customer Sales Performance ETL End-t...
14,Sales Data Warehouse and Analytics,Star-schema data warehouse and analytical repo...,"data warehouse, sql, tableau",https://github.com/sharlynmuturi/data-engineer...,https://sharlynmuturi.github.io/elements.html,Sales Data Warehouse and Analytics Star-schema...
7,SaaS Customer Churn Prediction and Retention,Developed an end-to-end SaaS customer churn pl...,"analysis, customer churn, machine learning, sh...",https://github.com/sharlynmuturi/machine-learn...,https://sharlynmuturi.github.io/generic.html,SaaS Customer Churn Prediction and Retention D...
4,Stock Forecasting & Portfolio Optimization,AI-driven system that applies machine learning...,"ai, ai agents, machine learning, prophet, stat...",https://github.com/sharlynmuturi/AI-Stock-Port...,https://sharlynmuturi.github.io/,Stock Forecasting & Portfolio Optimization AI-...


## Skill gap analysis

In [23]:
import re

job_skills = job_data.get("skills") or []
resume_skills = resume_data.get("skills") or []

In [24]:
# Extracting portfolio skills
portfolio_skills = set()

if top_projects_text:
    for line in top_projects_text.split("\n"):
        if line.startswith("Technologies:"):
            techs = line.replace("Technologies:", "").strip().split(",")
            for t in techs:
                t = t.strip()
                if t:
                    portfolio_skills.add(t)

portfolio_skills = list(portfolio_skills)

portfolio_skills

['sql',
 'prophet',
 'ai agents',
 'analysis',
 'survival analysis',
 'statistical methods',
 'ai',
 'analytics',
 'shap',
 'machine learning',
 'data warehouse',
 'customer churn',
 'visualization',
 'tableau']

In [25]:
combined_candidate_skills = resume_skills + portfolio_skills

combined_candidate_skills

['Python',
 'R',
 'SQL',
 'JavaScript',
 'HTML/CSS',
 'Django',
 'Flask',
 'Streamlit',
 'Databricks',
 'Tableau',
 'PowerBI',
 'SQL Server',
 'MySQL',
 'PostgreSQL',
 'Spark',
 'Excel',
 'Access',
 'Prompt Engineering',
 'LangChain',
 'RAG',
 'OpenAI',
 'Groq',
 'HuggingFace',
 'GenAI',
 'AgenticAI',
 'DocumentAI',
 'sql',
 'prophet',
 'ai agents',
 'analysis',
 'survival analysis',
 'statistical methods',
 'ai',
 'analytics',
 'shap',
 'machine learning',
 'data warehouse',
 'customer churn',
 'visualization',
 'tableau']

In [26]:
# Normalizing job skills
job_skills_normalized = set()

for skill in job_skills:
    if not skill:
        continue
        
    skill = skill.lower().strip()

    # Extract from parentheses
    paren_matches = re.findall(r'\((.*?)\)', skill)
    for match in paren_matches:
        for s in match.split(','):
            s = s.strip()
            if s:
                job_skills_normalized.add(s)

    # Remove parentheses part
    skill = re.sub(r'\(.*?\)', '', skill).strip()

    # Split remaining
    for s in skill.split(','):
        s = s.strip()
        if s:
            job_skills_normalized.add(s)

job_skills_normalized

{'c++',
 'data mining',
 'data visualization',
 'java',
 'javascript',
 'looker studio',
 'machine learning',
 'python',
 'r',
 'scala',
 'sql',
 'statistical analysis'}

In [27]:
candidate_skills_normalized = set()

for skill in combined_candidate_skills:
    if not skill:
        continue
        
    skill = skill.lower().strip()

    # Extract parentheses
    paren_matches = re.findall(r'\((.*?)\)', skill)
    for match in paren_matches:
        for s in match.split(','):
            s = s.strip()
            if s:
                candidate_skills_normalized.add(s)

    skill = re.sub(r'\(.*?\)', '', skill).strip()

    for s in skill.split(','):
        s = s.strip()
        if s:
            candidate_skills_normalized.add(s)

candidate_skills_normalized

{'access',
 'agenticai',
 'ai',
 'ai agents',
 'analysis',
 'analytics',
 'customer churn',
 'data warehouse',
 'databricks',
 'django',
 'documentai',
 'excel',
 'flask',
 'genai',
 'groq',
 'html/css',
 'huggingface',
 'javascript',
 'langchain',
 'machine learning',
 'mysql',
 'openai',
 'postgresql',
 'powerbi',
 'prompt engineering',
 'prophet',
 'python',
 'r',
 'rag',
 'shap',
 'spark',
 'sql',
 'sql server',
 'statistical methods',
 'streamlit',
 'survival analysis',
 'tableau',
 'visualization'}

In [28]:
# Computing Matches & Gaps
matched_skills_normalized = job_skills_normalized.intersection(candidate_skills_normalized)
missing_skills_normalized = job_skills_normalized.difference(candidate_skills_normalized)

matched_skills_normalized, missing_skills_normalized

({'javascript', 'machine learning', 'python', 'r', 'sql'},
 {'c++',
  'data mining',
  'data visualization',
  'java',
  'looker studio',
  'scala',
  'statistical analysis'})

In [29]:
with open("tailored_resume.txt", "w", encoding="utf-8") as f:
    f.write(tailored_resume.content)

In [30]:
with open("cover_letter.txt", "w", encoding="utf-8") as f:
    f.write(cover_letter_text)